# Phase 5 — Transaction Simulator

**Project:** Banking Transaction Monitoring & Fraud Analytics Platform
**Database:** `federal_bank`

This notebook reads `datasets/transactions.csv` (built in Phase 4) and inserts the
rows into the MySQL `transactions` table. It has **two modes**:

* **`bulk`** — loads every transaction quickly (so KPIs and the dashboard have data).
* **`live`** — inserts one transaction every 2–5 seconds to imitate a real-time feed.

**Why simulate real time?** Real banks ingest transactions continuously as a live
stream (often via message queues like Kafka feeding a database). We can't access a
live bank feed, so we *replay* our CSV: load history in bulk, then stream recent
transactions one at a time. This lets us demonstrate a live fraud-monitoring
pipeline — a strong talking point for interviews.

## 1. Configuration

In [ ]:
# --- MySQL connection (EDIT the password) ---
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_MYSQL_PASSWORD",   # <-- your MySQL password
    "database": "federal_bank",
}

TX_CSV_PATH = "datasets/transactions.csv"   # the feed created in Phase 4

# --- Choose how to run ---
MODE = "bulk"           # "bulk" = load everything fast | "live" = stream slowly (demo)

# Bulk-mode settings
TRUNCATE_FIRST = True   # clear the table before loading (makes bulk mode re-runnable)
BULK_BATCH     = 1000   # rows inserted per batch

# Live-mode settings
LIVE_LIMIT = 25         # how many recent transactions to stream
MIN_DELAY  = 2.0        # minimum seconds between streamed transactions
MAX_DELAY  = 5.0        # maximum seconds between streamed transactions

## 2. Imports

In [ ]:
import time
import random
import pandas as pd
import mysql.connector

## 3. Connect to MySQL

In [ ]:
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Connected to database:", DB_CONFIG["database"])

## 4. Read the transactions and the insert helpers

We read every column as **text** (`dtype=str`, `keep_default_na=False`) so long
numeric strings like account numbers aren't mangled into floating point, and blank
cells become `""`. We then convert each row into a tuple, turning blanks into `None`
(which MySQL stores as `NULL`).

In [ ]:
# SQL used for every insert. %s are placeholders filled safely by the connector
# (this also protects against SQL injection).
INSERT_SQL = (
    "INSERT INTO transactions "
    "(account_id, card_id, transaction_type, channel, amount, currency, status, "
    " transaction_city, counterparty_account, description, transaction_timestamp) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"
)


def load_records(csv_path):
    # Read the CSV and convert it into a list of tuples ready for INSERT.
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    records = []
    for r in df.to_dict("records"):
        records.append((
            int(r["account_id"]),
            int(float(r["card_id"])) if r["card_id"] else None,   # handles "266" and "266.0"; blank => NULL
            r["transaction_type"],
            r["channel"],
            float(r["amount"]),
            r["currency"],
            r["status"],
            r["transaction_city"] or None,
            r["counterparty_account"] or None,
            r["description"] or None,
            r["transaction_timestamp"],
        ))
    return records


def truncate_transactions(cursor, conn):
    # Empty the transactions table so a bulk load doesn't create duplicates.
    cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
    cursor.execute("TRUNCATE TABLE transactions")
    cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
    conn.commit()
    print("Cleared the transactions table.")


def bulk_load(records, cursor, conn, batch=1000):
    # Insert all rows quickly, in batches, committing after each batch.
    start = time.time()
    for i in range(0, len(records), batch):
        cursor.executemany(INSERT_SQL, records[i:i + batch])
        conn.commit()
    print(f"Bulk-loaded {len(records)} transactions in {time.time() - start:.1f}s.")


def live_stream(records, cursor, conn, limit=25, min_delay=2.0, max_delay=5.0):
    # Stream the most-recent `limit` transactions one at a time, like a live feed.
    stream = records[-limit:]
    print(f"Streaming {len(stream)} transactions (one every {min_delay}-{max_delay}s)...\n")
    for i, rec in enumerate(stream, 1):
        cursor.execute(INSERT_SQL, rec)
        conn.commit()
        # rec = (account_id, card_id, type, channel, amount, currency, status, city, cp, desc, ts)
        print(f"[{i}/{len(stream)}] {rec[10]} | acct {rec[0]:>4} | "
              f"{rec[3]:<8} {rec[2]:<10} INR {rec[4]:>10.2f} | {rec[6]}")
        if i < len(stream):
            time.sleep(random.uniform(min_delay, max_delay))
    print("\nLive stream complete.")

## 5. Load the records

In [ ]:
records = load_records(TX_CSV_PATH)
print(f"Loaded {len(records)} transactions from {TX_CSV_PATH}")

## 6. Run the simulator

Set `MODE` in the Configuration cell:
* `"bulk"` populates the whole table (run this first so later phases have data).
* `"live"` streams the most recent `LIVE_LIMIT` transactions slowly, printing each.

> Note: running `live` after `bulk` re-inserts that small recent subset (harmless for
> a demo). Re-running `bulk` (with `TRUNCATE_FIRST = True`) resets to a clean load.

In [ ]:
if MODE == "bulk":
    if TRUNCATE_FIRST:
        truncate_transactions(cursor, conn)
    bulk_load(records, cursor, conn, batch=BULK_BATCH)
elif MODE == "live":
    live_stream(records, cursor, conn, limit=LIVE_LIMIT,
                min_delay=MIN_DELAY, max_delay=MAX_DELAY)
else:
    print("Set MODE to 'bulk' or 'live' in the Configuration cell.")

## 7. Verify

In [ ]:
cursor.execute("SELECT COUNT(*) FROM transactions")
print("Rows now in the transactions table:", cursor.fetchone()[0])

cursor.execute("SELECT status, COUNT(*) FROM transactions GROUP BY status")
for status, count in cursor.fetchall():
    print(f"  {status:<9}: {count}")

## 8. Close the connection

In [ ]:
cursor.close()
conn.close()
print("Done. The transactions table is now populated and ready for Phase 6 (ETL).")